# Data Modeling - Affect Juncion Table

### Affect Table

To represent the many-to-many relationship between the `interruption` and `location` tables, a junction table is required to map their relationships.

This table includes:
* `location_id` – a foreign key referencing the `location` table.  
* `interruption_id` – a foreign key referencing the `interruption` table.  

The junction table is created through a series of transformations:

* **Concatenate barangay values**  
The barangay column from the `location_gold` table is concatenated into a regex pattern and used with `regexp_extract_all` to capture matching locations within the text column.

* **Handle ambiguous location names**  
Some locations (e.g., *Balili*, *Poblacion*) exist in multiple municipalities. These are distinguished by appending a municipality abbreviation (e.g., `Poblacion TB`, `Poblacion SB`).  
Since the abbreviation is not always included in the posts, an additional regex extraction step is applied to capture these cases.
* **Handle unmatched locations**  
Some locations mentioned in posts are not present in the BENECO locaiton table and therefore cannot be captured by the current matching logic.


In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.affect_temp (
  id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  text STRING,
  matched STRING
);
INSERT INTO TABLE beneco_pipeline.gold.affect_temp(text, matched)
WITH temp AS (
  SELECT 
    concat('(?i)\\b(',concat_ws('|', collect_list(barangay)),')\\b') AS brgy
  FROM beneco_pipeline.silver.location_silver
)
SELECT 
  text,
  CASE 
    WHEN text RLIKE brgy THEN (
      array_join(regexp_extract_all(text, brgy, 0),', ')
    )
    WHEN text RLIKE 'Ambiong' THEN (
      CASE 
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Ambiong',1) RLIKE '(?i)La Trinidad' THEN 'Ambiong LT'
        ELSE 'Ambiong BC'
      END
    )
    WHEN text RLIKE 'Balili' THEN (
      CASE 
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Balili',1) RLIKE '(?i)La Trinidad' THEN 'Balili LT'
        ELSE 'Balili MK'
      END
    )
    WHEN text RLIKE 'Poblaci(o|ó)n' THEN (
      CASE 
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Atok' THEN 'Poblacion AT'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Bokod' THEN 'Poblacion BD'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Bakun' THEN 'Poblacion BN'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Buguias' THEN 'Poblacion BS'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Kapangan' THEN 'Poblacion Central'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Itogon' THEN 'Poblacion IT'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)La Trinidad' THEN 'Poblacion LT'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Mankayan' THEN 'Poblacion MK'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Sablan' THEN 'Poblacion SB'
        WHEN regexp_extract(text, r'(?i)(\w+(\s\w+)?):.+Poblaci(?:o|ó)n',1) RLIKE '(?i)Tuba' THEN 'Poblacion TB'
        ELSE 'Poblacion KB'
      END
    )    
    ELSE 'not matched'
  END AS matched_names
FROM beneco_pipeline.silver.interruption_silver AS ben
CROSS JOIN temp;

### Location Aliases
Certain locations may be written differently in posts compared to their official barangay names from the BENECO know your feeder dataset. To address this a new table is made containing the different aliases a location can have. These aliases are then used to map the locations in the `interruptions_gold` text column to the locations in the `location_gold` barangay column.

The following alias table contains some of the location name aliases of the locations not captured in the previous transformation.


In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.alias (
    alias STRING,
    loc_name STRING
);
INSERT INTO beneco_pipeline.gold.alias VALUES
('camp allen', 'Camp Henry I. Allen'),
('Saint Joseph','St. Joseph Village'),
('Saint Joseph Village','St. Joseph Village'),
('Bakakeng Norte','Bakakeng Norte/Sur'),
('Bakakeng Sur','Bakakeng Norte/Sur'),
('Bayan Park','Bayan Park Village'),
('Ambuclao','Ambuklao'),
('Sto. Nino- Slaughter Compound','Sto. Nino-Slaughter Compound'),
('Apugan-Loakan','Loakan Apugan'),
('Sto Tomas Proper','Sto. Tomas Proper'),
('San Luis', 'San Luis Village');


Multiple aliases may match a given location in the outage posts. In such cases, the match with the greatest character length is selected, as it is assumed to be the most accurate. If the `affect_temp` table has no matches and the alias matches the text column, the `affect_temp.match` column will be replaced with the location name assoicated with the matched alias.

For example, `Saint Joseph Village` matches both the aliases `Saint Joseph` and `Saint Joseph Village`. The latter is chosen because it is more specific and provides a more complete representation of the location.

In [0]:
%skip
%sql
SELECT * FROM beneco_pipeline.gold.affect_temp WHERE matched = 'not matched';
SELECT 
  at.text,
  a.loc_name,
  -- ranking by match length
  ROW_NUMBER() OVER (PARTITION BY at.text ORDER BY LENGTH(a.alias) DESC) AS rn
FROM beneco_pipeline.gold.affect_temp AS at
CROSS JOIN beneco_pipeline.gold.alias AS a
WHERE at.text RLIKE concat('(?i)\\b(',a.alias,')\\b')
  AND at.matched = 'not matched';

In [0]:
%sql
WITH ranked_matches AS (
  SELECT 
    at.text,
    a.loc_name,
    -- ranking by match length
    DENSE_RANK() OVER (PARTITION BY at.text ORDER BY LENGTH(a.alias)) AS rn
  FROM beneco_pipeline.gold.affect_temp AS at
  CROSS JOIN beneco_pipeline.gold.alias AS a
  WHERE at.text RLIKE concat('(?i)\\b(',a.alias,')\\b') 
    AND at.matched = 'not matched'
)
--Merge first rank into affect_temp
MERGE INTO beneco_pipeline.gold.affect_temp AS at
USING (
  SELECT 
    text, 
    loc_name 
    FROM ranked_matches
    -- best match 
    WHERE rn = 1) 
    AS a
ON at.matched = 'not matched' AND at.text = a.text
WHEN MATCHED THEN
  UPDATE SET at.matched = a.loc_name;

--Query for not matched locations
SELECT * FROM beneco_pipeline.gold.affect_temp WHERE matched = 'not matched';
  

# Create affect_gold Table
The affected_gold table is created to store the relationship between interruptions and locations using their respective primary keys.  

To populate this table:
* The transformed dataset containing matched locations from the post text is joined with the interruption_gold and location_gold tables.
* The corresponding interruption_id and location_id are selected to establish the relationship.
* Foreign key constraints are defined to enforce referential integrity.

In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.affect_gold(
  interruption_id BIGINT,
  location_id BIGINT,
  FOREIGN KEY (location_id) REFERENCES beneco_pipeline.gold.location_gold(id),
  FOREIGN KEY (interruption_id) REFERENCES beneco_pipeline.gold.interruption_gold(id)
);

INSERT INTO beneco_pipeline.gold.affect_gold
SELECT 
  i.id AS interruption_id,
  l.id AS location_id
FROM (
  SELECT 
    id,
    text,
    explode(split(trim(matched), ', ')) AS match
  FROM beneco_pipeline.gold.affect_temp
) AS at
JOIN beneco_pipeline.gold.interruption_gold AS i ON 
  i.text = at.text
JOIN beneco_pipeline.gold.location_gold AS l ON
  l.barangay = at.match;

In [0]:
%sql
-- WITH TEMP as (
SELECT t.municipality, count(t.municipality) AS total
FROM(SELECT DISTINCT
  a.interruption_id AS interruption_id,
  l.municipality AS municipality
FROM beneco_pipeline.gold.affect_gold AS a
JOIN beneco_pipeline.gold.location_gold AS l
  ON a.location_id = l.id
ORDER BY interruption_id) AS t
GROUP BY t.municipality
-- )SELECT SUM(total) FROM TEMP
;
